In [1]:
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import random
import itertools
import networkx as nx
from collections import defaultdict
from sklearn.model_selection import train_test_split


In [2]:
SAMPLED_DATA_DIR = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/8"
SEED             = 42

train_path = os.path.join(SAMPLED_DATA_DIR, "hm_subset_train.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, "hm_subset_test.csv")

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)

# Cast IDs to str to handle mixed-type H&M IDs consistently
for df in [train_df, test_df]:
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)

# Build contiguous 0-based encoders fitted on train only
user_enc = {u: i for i, u in enumerate(sorted(train_df["user_id"].unique()))}
item_enc = {v: i for i, v in enumerate(sorted(train_df["item_id"].unique()))}

def remap(df, user_enc, item_enc, drop_unknown=False):
    df = df.copy()
    if drop_unknown:
        df = df[df["user_id"].isin(user_enc) & df["item_id"].isin(item_enc)]
    df["user_id"] = df["user_id"].map(user_enc)
    df["item_id"] = df["item_id"].map(item_enc)
    return df.dropna(subset=["user_id", "item_id"]).astype({"user_id": int, "item_id": int})

train_df = remap(train_df, user_enc, item_enc)
test_df  = remap(test_df,  user_enc, item_enc, drop_unknown=True)

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=0)

n_users = len(user_enc)
n_items = len(item_enc)
print(f"Total Users : {n_users}")
print(f"Total Items : {n_items}")
print(f"Train : {len(train_df)} | Val : {len(val_df)} | Test : {len(test_df)}")
train_df.head()


Total Users : 1760
Total Items : 2881
Train : 62568 | Val : 15643 | Test : 19553


,user_id,item_id,bought
17812,23,2123,1
788,491,714,1
1399,357,1741,1
12060,590,1828,1
31118,1248,1025,1


In [3]:
# ── Build per-user train / val / test dicts ───────────────────────────────────
def df_to_dict(df):
    d = {}
    for uid, grp in df.groupby("user_id"):
        # columns: user_id, item_id, bought  → cast to numpy array (n,3)
        d[int(uid)] = grp[["user_id", "item_id", "bought"]].values
    return d

train_data_dict = df_to_dict(train_df)
val_data_dict   = df_to_dict(val_df)
test_data_dict  = df_to_dict(test_df)

num_users = n_users
num_items = n_items

# Only keep users present in train AND test
common_users = set(train_data_dict.keys()) & set(test_data_dict.keys())
train_data_dict = {u: v for u, v in train_data_dict.items() if u in common_users}
val_data_dict   = {u: v for u, v in val_data_dict.items()   if u in common_users}
test_data_dict  = {u: v for u, v in test_data_dict.items()  if u in common_users}

total_train = sum(len(v) for v in train_data_dict.values())
total_test  = sum(len(v) for v in test_data_dict.values())
print(f"Active users (train∩test) : {len(common_users)}")
print(f"Train interactions        : {total_train}")
print(f"Val   interactions        : {sum(len(v) for v in val_data_dict.values())}")
print(f"Test  interactions        : {total_test}")
print(f"Density: {(total_train + total_test) / (num_users * num_items) * 100:.4f}%")


Active users (train∩test) : 1760
Train interactions        : 62568
Val   interactions        : 15643
Test  interactions        : 19553
Density: 1.6196%


In [4]:
# ── Communication graph (random-2-out via networkx) ───────────────────────────
active_ids = sorted(common_users)
np.random.seed(SEED)
G = nx.DiGraph()
G.add_nodes_from(active_ids)
for u in active_ids:
    candidates = [v for v in active_ids if v != u]
    targets    = np.random.choice(candidates, size=min(2, len(candidates)), replace=False)
    for t in targets:
        G.add_edge(u, t)
G = G.to_undirected()
print(f"Graph : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"Avg degree : {np.mean([d for _, d in G.degree()]):.2f}")


Graph : 1760 nodes, 3518 edges
Avg degree : 4.00


In [5]:
# ── Model & helper definitions ────────────────────────────────────────────────

def initX(k, R_min=0.0, R_max=1.0):
    scale = np.sqrt((R_max - R_min) / k)
    x = torch.rand(k) * scale
    b = torch.tensor(R_min / 2.0)
    return x, b

def initY(num_items, k, R_min=0.0, R_max=1.0):
    Y = torch.zeros(num_items, k)
    c = torch.zeros(num_items)
    t = torch.zeros(num_items)
    for j in range(num_items):
        Y[j], c[j] = initX(k, R_min, R_max)
    return Y, c, t


class RatingDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        user, item, rating = self.data[idx]
        return user, item, rating.float()


class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=20):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.bias     = nn.Parameter(torch.zeros(num_items))
    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        return (u * i).sum(1) + self.bias[items]


def compress_model(Y, c, t, compression_ratio):
    num_items = Y.shape[0]
    k = max(1, int(compression_ratio * num_items))
    indices = np.random.choice(num_items, k, replace=False)
    return indices, Y[indices], c[indices], t[indices]

def merge_model(Y, c, t, Y_recv, c_recv, t_recv, indices):
    for i, j in enumerate(indices):
        if t_recv[i] > 0:
            tj, t_new = t[j].item(), t_recv[i].item()
            w    = t_new / (tj + t_new) if (tj + t_new) > 0 else 0.5
            t[j] = max(tj, t_new)
            Y[j] = (1 - w) * Y[j] + w * Y_recv[i]
            c[j] = (1 - w) * c[j] + w * c_recv[i]
    return Y, c, t

def train_local(model, dataloader, optimizer, criterion, device):
    model.train()
    for users, items, ratings in dataloader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        preds = model(users, items)
        loss  = criterion(preds, ratings)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def evaluate(model, dataloader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for users, items, ratings in dataloader:
            users, items = users.to(device), items.to(device)
            pred = model(users, items)
            preds.extend(pred.cpu().numpy())
            trues.extend(ratings.numpy())
    return np.sqrt(np.mean((np.array(preds) - np.array(trues)) ** 2))


def gossip_learning(num_users, num_items, train_data_dict, eval_data_dict,
                    graph, epochs=10, emb_dim=20, lr=0.01,
                    compression_ratio=0.1, patience=10):
    """
    Runs gossip learning and returns mean RMSE over eval_data_dict.
    Pass val_data_dict for tuning; test_data_dict for final evaluation.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    user_models, user_Y, user_c, user_t = {}, {}, {}, {}
    train_loaders, eval_loaders, optimizers = {}, {}, {}
    criterion = nn.MSELoss()

    for u in graph.nodes():
        if u not in train_data_dict:
            continue
        model = MF(num_users, num_items, emb_dim).to(device)
        user_models[u]  = model
        optimizers[u]   = optim.Adam(model.parameters(), lr=lr)
        Y, c, t         = initY(num_items, emb_dim)
        user_Y[u], user_c[u], user_t[u] = Y.clone(), c.clone(), t.clone()
        model.item_emb.weight.data = Y.clone()
        model.bias.data            = c.clone()
        train_loaders[u] = DataLoader(RatingDataset(train_data_dict[u]),
                                      batch_size=64, shuffle=True)
        if u in eval_data_dict:
            eval_loaders[u] = DataLoader(RatingDataset(eval_data_dict[u]),
                                         batch_size=256)

    active      = list(user_models.keys())
    best_rmse   = float('inf')
    no_improve  = 0

    for epoch in range(epochs):
        for u in active:
            train_local(user_models[u], train_loaders[u], optimizers[u],
                        criterion, device)
        for u in active:
            neighbors = [v for v in graph.neighbors(u) if v in user_models]
            if not neighbors:
                continue
            partner = random.choice(neighbors)
            indices, Y_s, c_s, t_s = compress_model(
                user_Y[u], user_c[u], user_t[u], compression_ratio)
            user_Y[partner], user_c[partner], user_t[partner] = merge_model(
                user_Y[partner], user_c[partner], user_t[partner],
                Y_s, c_s, t_s, indices)
            user_models[partner].item_emb.weight.data = user_Y[partner]
            user_models[partner].bias.data            = user_c[partner]

        # Early stopping on eval RMSE
        eval_users = [u for u in active if u in eval_loaders]
        if eval_users:
            rmse = np.mean([evaluate(user_models[u], eval_loaders[u], device)
                            for u in eval_users])
            if rmse < best_rmse:
                best_rmse, no_improve = rmse, 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    break

    eval_users = [u for u in active if u in eval_loaders]
    if eval_users:
        return np.mean([evaluate(user_models[u], eval_loaders[u], device)
                        for u in eval_users])
    return float('nan')

print("gossip_learning and helpers defined.")


gossip_learning and helpers defined.


In [6]:
# ── Parameter Tuning — Grid Search ───────────────────────────────────────────
EMB_GRID  = [10,   20,   40  ]
LR_GRID   = [0.001, 0.01, 0.05]
CR_GRID   = [0.1,  0.3       ]   # compression ratio
TUNE_EP   = 30                   # epochs per combo (early-stop inside)
TUNE_PAT  = 5

param_grid = list(itertools.product(EMB_GRID, LR_GRID, CR_GRID))
print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EP} epochs")
print(f"{'#':>3}  {'emb':>5}  {'lr':>7}  {'cr':>5}  {'best val RMSE':>14}")
print("-" * 42)

gs_results  = []
best_rmse   = float('inf')
best_config = None

for _k, (emb, lr, cr) in enumerate(param_grid, 1):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(0)
    _v = gossip_learning(
        num_users, num_items,
        train_data_dict, val_data_dict,
        G,
        epochs=TUNE_EP, emb_dim=emb, lr=lr,
        compression_ratio=cr, patience=TUNE_PAT)
    gs_results.append((emb, lr, cr, _v))
    if _v < best_rmse:
        best_rmse   = _v
        best_config = (emb, lr, cr)
    _marker = ' <--' if _v == best_rmse else ''
    print(f"{_k:>3}  {emb:>5d}  {lr:>7.4f}  {cr:>5.2f}  {_v:>14.4f}{_marker}")

print(f"\nBest val RMSE  : {best_rmse:.4f}")
print(f"  emb_dim        = {best_config[0]}")
print(f"  lr             = {best_config[1]}")
print(f"  compression    = {best_config[2]}")
emb_dim           = best_config[0]
lr                = best_config[1]
compression_ratio = best_config[2]
print("\nHyperparameters updated. Run the training cell.")


Grid search: 18 combinations x up to 30 epochs
  #    emb       lr     cr   best val RMSE
------------------------------------------
  1     10   0.0010   0.10          1.6009 <--
  2     10   0.0010   0.30          1.6009 <--
  3     10   0.0100   0.10          1.3802 <--
  4     10   0.0100   0.30          1.3802 <--
  5     10   0.0500   0.10          1.2120 <--
  6     10   0.0500   0.30          1.2120 <--
  7     20   0.0010   0.10          1.5787
  8     20   0.0010   0.30          1.5787
  9     20   0.0100   0.10          1.3930
 10     20   0.0100   0.30          1.3930
 11     20   0.0500   0.10          1.2084 <--
 12     20   0.0500   0.30          1.2084 <--
 13     40   0.0010   0.10          1.5540
 14     40   0.0010   0.30          1.5540
 15     40   0.0100   0.10          1.4235
 16     40   0.0100   0.30          1.4235
 17     40   0.0500   0.10          1.1951 <--
 18     40   0.0500   0.30          1.1951 <--

Best val RMSE  : 1.1951
  emb_dim        = 40
  lr  

In [7]:
# ── Retrain best config, evaluate on test ────────────────────────────────────
try: emb_dim
except NameError: emb_dim = 20
try: lr
except NameError: lr = 0.01
try: compression_ratio
except NameError: compression_ratio = 0.1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(0)
test_rmse = gossip_learning(
    num_users, num_items,
    train_data_dict, test_data_dict,
    G,
    epochs=100, emb_dim=emb_dim, lr=lr,
    compression_ratio=compression_ratio, patience=10)

print(f"Test RMSE : {test_rmse:.4f}")


Test RMSE : 1.2080


In [8]:
# ── Grid search results summary ───────────────────────────────────────────────
gs_df = pd.DataFrame(gs_results,
                     columns=["emb_dim", "lr", "compression_ratio", "val_rmse"])
print("Best config:")
print(gs_df.loc[gs_df["val_rmse"].idxmin()])
gs_df


Best config:
emb_dim              40.00000
lr                    0.05000
compression_ratio     0.10000
val_rmse              1.19509
Name: 16, dtype: float64


,emb_dim,lr,compression_ratio,val_rmse
0,10,0.001,0.1,1.600940
1,10,0.001,0.3,1.600940
2,10,0.010,0.1,1.380187
3,10,0.010,0.3,1.380187
4,10,0.050,0.1,1.212023
5,10,0.050,0.3,1.212023
6,20,0.001,0.1,1.578734
7,20,0.001,0.3,1.578734
8,20,0.010,0.1,1.393014
9,20,0.010,0.3,1.393014
